# Modèle Word2Vec et similarité cosinus

### Lecture du dataset nettoyé

In [2]:
import pandas as pd

df = pd.read_csv("../../datasets/clean/perfumes_clean.csv")

In [3]:
df.head()


,id,name,house,url,family,secondary_family,character,top_notes,heart_notes,base_notes,...,times_of_day,season_spring,season_summer,season_autumn,season_winter,time_day,time_evening,time_night,description,text_full
0,ysl-libre,Libre,Yves Saint Laurent,https://www.scento.com/perfumes/libre-yves-sai...,White Bloom,NaN,floral; creamy; fresh; citrusy,mandarin; neroli; lavender,orange blossom; jasmine sambac,musk; vanilla,...,day; evening,1,1,1,0,1,1,0,Mandarin and neroli brighten a creamy bloom of...,Mandarin and neroli brighten a creamy bloom of...
1,guerlain-shalimar,Shalimar,Guerlain,https://www.scento.com/perfumes/shalimar-eau-d...,Petal Garden,Velvet Sweet,floral; sweet; warm; powdery,floral notes; bergamot,iris; jasmine; rose,vanilla; opoponax; tonka bean,...,day; evening,1,1,1,0,1,1,0,Floral notes and a flash of bergamot sit above...,Floral notes and a flash of bergamot sit above...
2,tom-ford-oud-wood,Oud Wood,Tom Ford,https://www.scento.com/perfumes/oud-wood-tom-ford,Spice Route,NaN,warm; spicy; sweet; earthy,NaN,NaN,NaN,...,evening; night,0,0,1,1,0,1,1,Smooth woods and spice: cardamom and pink pepp...,Smooth woods and spice: cardamom and pink pepp...
3,creed-aventus,Aventus,Creed,https://www.scento.com/perfumes/aventus-creed,Orchard Fruit,Wood & Root,fruity; earthy; citrusy; floral,lemon; pink pepper; apple; bergamot; blackcurrant,pineapple; jasmine; patchouli,birch; ambroxan; cedarwood; oakmoss; musk,...,day; evening,1,1,1,0,1,1,0,Sparkling lemon and blackcurrant open onto pin...,Sparkling lemon and blackcurrant open onto pin...
4,mfk-baccarat-rouge-540,Baccarat Rouge 540,Maison Francis Kurkdjian,https://www.scento.com/perfumes/baccarat-rouge...,Wood & Root,White Bloom,woody; warm; floral; spicy,jasmine; saffron,cedarwood,ambergris,...,day; evening,1,0,1,1,1,1,0,"Jasmine and saffron open airy and golden, held...","Jasmine and saffron open airy and golden, held..."


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1479 entries, 0 to 1478
Data columns (total 24 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   id                1479 non-null   object 
 1   name              1479 non-null   object 
 2   house             1479 non-null   object 
 3   url               1479 non-null   object 
 4   family            1479 non-null   object 
 5   secondary_family  1075 non-null   object 
 6   character         1479 non-null   object 
 7   top_notes         1388 non-null   object 
 8   heart_notes       1388 non-null   object 
 9   base_notes        1388 non-null   object 
 10  notes             1479 non-null   object 
 11  concentration     1479 non-null   object 
 12  intensity         1228 non-null   float64
 13  seasons           1479 non-null   object 
 14  times_of_day      1479 non-null   object 
 15  season_spring     1479 non-null   int64  
 16  season_summer     1479 non-null   int64  


Le dataset propre contient désormais 24 colonnes. Les colonnes de saisons et de moments de la journée sont en binaires, donc seront exploitables dans le questionnaire lors de la création de l'interface. La colonne text_full contient toutes les données de la ligne dans une colonne.

### Tokénisation

On commence par l'étape de tokénisation. Word2Vec nécessite des données sous forme de listes de mots et non de phrase complète. Il s'agit donc de découper chaque description de la colonne texte_full en liste de mots. Pour plus d'uniformité, nous allons aussi en profité pour mettre tous les caractères en minuscule. 

Après une première vérification des résultats, certains mots étaient tokénisés avec des virgules ou des points. On retire donc également la ponctuation.

In [5]:
# découpage du texte en mot
corpus = df['text_full'].apply(lambda texte: texte.lower().split())

# retirage de la ponctuation
corpus = df["text_full"].apply(lambda texte: texte.lower().replace(",", "").replace(".", "").replace(":", "").split())

print('Texte original : ', df['text_full'].iloc[0])
print('Texte tokénisé : ', corpus.iloc[0])

Texte original :  Mandarin and neroli brighten a creamy bloom of orange blossom and jasmine sambac, settling into a quiet base of musk and vanilla. Olfactory family: White Bloom. Character: floral, creamy, fresh, citrusy. Top notes: mandarin, neroli, lavender. Heart notes: orange blossom, jasmine sambac. Base notes: musk, vanilla. Seasons: spring, summer, autumn. Time of day: day, evening.
Texte tokénisé :  ['mandarin', 'and', 'neroli', 'brighten', 'a', 'creamy', 'bloom', 'of', 'orange', 'blossom', 'and', 'jasmine', 'sambac', 'settling', 'into', 'a', 'quiet', 'base', 'of', 'musk', 'and', 'vanilla', 'olfactory', 'family', 'white', 'bloom', 'character', 'floral', 'creamy', 'fresh', 'citrusy', 'top', 'notes', 'mandarin', 'neroli', 'lavender', 'heart', 'notes', 'orange', 'blossom', 'jasmine', 'sambac', 'base', 'notes', 'musk', 'vanilla', 'seasons', 'spring', 'summer', 'autumn', 'time', 'of', 'day', 'day', 'evening']


In [6]:
print("Nombre de documents :", len(corpus))
print("Nombre de mots du premier parfum :", len(corpus.iloc[0]))

longueurs = corpus.apply(len)
print("Nombre minimum de mots :", longueurs.min())
print("Nombre maximum de mots :", longueurs.max())
print("Nombre moyen de mots :", round(longueurs.mean(), 2))

Nombre de documents : 1479
Nombre de mots du premier parfum : 55
Nombre minimum de mots : 29
Nombre maximum de mots : 86
Nombre moyen de mots : 58.48


Après tokénisation, on obtient 1479 corpus, soit un par parfum. Chaque corpus contient en moyenne 58,48 mots, avec 29 en minimum et 86 en maximum

### Suppression des stop words

Comme vu en cours, l'étape suivante est de retirer les stop words pour ne pas garder des mots inutiles qui ne joue pas sur le sens de la phrase.

In [7]:
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

corpus = corpus.apply(lambda mots: [mot for mot in mots if mot not in ENGLISH_STOP_WORDS])

In [8]:
print('Texte après suppression des stop words : ', corpus.iloc[0])

print('Moyenne de mots avant : ', round(longueurs.mean(), 2))
print('Moyenne de mots après : ', round(corpus.apply(len).mean(), 2))

Texte après suppression des stop words :  ['mandarin', 'neroli', 'brighten', 'creamy', 'bloom', 'orange', 'blossom', 'jasmine', 'sambac', 'settling', 'quiet', 'base', 'musk', 'vanilla', 'olfactory', 'family', 'white', 'bloom', 'character', 'floral', 'creamy', 'fresh', 'citrusy', 'notes', 'mandarin', 'neroli', 'lavender', 'heart', 'notes', 'orange', 'blossom', 'jasmine', 'sambac', 'base', 'notes', 'musk', 'vanilla', 'seasons', 'spring', 'summer', 'autumn', 'time', 'day', 'day', 'evening']
Moyenne de mots avant :  58.48
Moyenne de mots après :  48.3


Avec la suppression des stop words, on finit avec 48,3 mots en moyenne au liey de 58,48 avant.

### Suppression des mots de structure

On retire également les mots de structure pour ne pas fausser l'apprentissage des descriptions.

In [9]:
mots_inutiles = ['olfactory', 'family', 'character', 'notes', 'seasons', 'time', 'day', 'heart', 'base']

corpus = corpus.apply(lambda mots: [mot for mot in mots if mot not in mots_inutiles])

In [10]:
print(corpus.iloc[0])
print('Moyenne de mots après nettoyage : ', round(corpus.apply(len).mean(), 2))

['mandarin', 'neroli', 'brighten', 'creamy', 'bloom', 'orange', 'blossom', 'jasmine', 'sambac', 'settling', 'quiet', 'musk', 'vanilla', 'white', 'bloom', 'floral', 'creamy', 'fresh', 'citrusy', 'mandarin', 'neroli', 'lavender', 'orange', 'blossom', 'jasmine', 'sambac', 'musk', 'vanilla', 'spring', 'summer', 'autumn', 'evening']
Moyenne de mots après nettoyage :  35.83


On passe désormais à 35,83 mots en moyenne.

### Vérification du vocabulaire

In [11]:
from collections import Counter

# on compte les occurrences de chaque mot
frequences = Counter(mot for parfum in corpus for mot in parfum)

print('Nombre de mots différents : ', len(frequences))
print('\nLes 20 mots les plus fréquents : ')
print(frequences.most_common(20))

Nombre de mots différents :  1239

Les 20 mots les plus fréquents : 
[('evening', 1445), ('autumn', 1442), ('spring', 1415), ('warm', 1192), ('floral', 1176), ('musk', 1171), ('vanilla', 1059), ('sweet', 997), ('bergamot', 992), ('amber', 949), ('summer', 880), ('patchouli', 867), ('jasmine', 866), ('rose', 832), ('sandalwood', 748), ('wood', 701), ('orange', 672), ('fresh', 672), ('cedarwood', 622), ('&', 617)]


In [12]:
mots_uniques = sum(1 for frequence in frequences.values() if frequence == 1)

print('Mots apparaissant une seule fois : ', mots_uniques)
print("Pourcentage de mot apparaissant qu'une seule fois : ", round(mots_uniques / len(frequences) * 100, 2), "%")

Mots apparaissant une seule fois :  407
Pourcentage de mot apparaissant qu'une seule fois :  32.85 %


Certains mots sont très fréquents, mais 32,8% n'apparaissent qu'une seule fois. Il peut donc être intéressant de quand même garder les mots rares pour la vectorisation.

### Entraînement de Word2Vec

In [13]:
from gensim.models import Word2Vec

model = Word2Vec(
    sentences=corpus,
    vector_size=100,
    window=5,
    min_count=1, # on garde tous les mots
    workers=4,
    sg=1,
    epochs=100
)

/Users/lisaroubertieperso/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


Nous voulons pouvoir établir des relations entre tous les mots, même ceux qui sont uniques. Nous allons donc travailler avec min_count=1, pour garder les termes spécifiques. Néanmoins, il faut noter que cela peut inclure des mots peu informatifs.

In [14]:
print('Nombre de mots dans le modèle : ', len(model.wv))
print('Dimension des vecteurs : ', model.wv.vector_size)

Nombre de mots dans le modèle :  1239
Dimension des vecteurs :  100


On retrouve 1239 mots et des vecteurs de 100 dimensions.

In [15]:
model.wv.most_similar('jasmine', topn=10)

[('patchouli', 0.5830302238464355),
 ('rose', 0.5526590943336487),
 ('vanilla', 0.5259930491447449),
 ('jammy', 0.5199525356292725),
 ('musk', 0.5159407258033752),
 ('berry-floral', 0.4869513213634491),
 ('elder', 0.485904335975647),
 ('bergamot', 0.48497191071510315),
 ('float', 0.48269397020339966),
 ('playfully', 0.4824671745300293)]

In [16]:
model.wv.most_similar('musk', topn=10)

[('amber', 0.5731196999549866),
 ('sandalwood', 0.5563965439796448),
 ('vanilla', 0.5521451234817505),
 ('jasmine', 0.5159407258033752),
 ('patchouli', 0.49870142340660095),
 ('transparent', 0.48991522192955017),
 ('easing', 0.4852370321750641),
 ('kind', 0.47693923115730286),
 ('fennel', 0.47194716334342957),
 ('wood', 0.47155269980430603)]

In [17]:
model.wv.most_similar('vanilla', topn=10)

[('sweet', 0.6184694170951843),
 ('amber', 0.5903140902519226),
 ('bean', 0.5659189224243164),
 ('sandalwood', 0.5645151138305664),
 ('patchouli', 0.5619387626647949),
 ('musk', 0.5521451234817505),
 ('tonka', 0.5377005338668823),
 ('amber-toned', 0.5270102024078369),
 ('jasmine', 0.5259930491447449),
 ('appear', 0.518679678440094)]

Les scores de similarité varient entre 0,5 et 0,6 environ. Par exemple, pour jasmine, les senteurs proches sont patchouli, jasmin et orange. Pour vanilla, le musk et l'ambre sont proches. La vanille est d'ailleurs retrouvé aussi dans le top 10 du musk, ce qui est cohérent.

### Création des vecteurs de parfums

In [18]:
import numpy as np

def moyenne_parfum(liste):
    vecteurs = [model.wv[mot] for mot in liste]
    return np.mean(vecteurs, axis=0)

On commence par faire une moyenne des vecteurs de chaque mots d'une ligne.

In [19]:
vecteur_test = moyenne_parfum(corpus.iloc[0])

print(vecteur_test)
print('Dimension du vecteur : ', vecteur_test.shape)

[-0.1438222   0.26841015 -0.10636626  0.06219242  0.14076424 -0.13796444
  0.3250185   0.22169156 -0.10754432 -0.09887847 -0.33071202 -0.06879379
  0.09054516  0.06362258  0.09907863 -0.21685044 -0.00400977 -0.21412621
  0.16526577 -0.24579853  0.29256198 -0.149955    0.12113439  0.0329667
  0.03339219 -0.01645957 -0.09374698 -0.23725545 -0.20850877  0.10424364
  0.09426686 -0.23988888  0.13805741 -0.20499012 -0.13693593  0.22640161
 -0.04273523  0.07523917 -0.03314195 -0.26012626  0.13646148 -0.3887293
 -0.02836803  0.16250964  0.20872901 -0.35509652 -0.04370592 -0.31652296
 -0.04322963  0.02236518  0.11142537 -0.11186015 -0.03658136 -0.07094602
 -0.10059469  0.03070768  0.08600187 -0.07599226 -0.14836366 -0.02281894
  0.04116147 -0.24041192  0.01981118 -0.12552767 -0.02127086  0.10282099
 -0.02436537  0.20967081 -0.24090667  0.16291618 -0.11438158 -0.0806554
  0.22190623  0.12128889  0.24153422  0.0893883   0.02365966  0.13775377
 -0.09513941  0.04068351 -0.06916376 -0.0576478  -0.36

La fonction semble fonctionner : on obtient bien un vecteur de 100 nombres

In [20]:
# On applique cela à tout les parfums

vecteurs_parfums = np.array([
    moyenne_parfum(mots) for mots in corpus
])

print("Dimensions de la matrice :", vecteurs_parfums.shape)

Dimensions de la matrice : (1479, 100)


Après application sur tout les parfums, on fini bien avec 1479 vecteurs de dimension 100, soit un vecteur par lignes du dataset.

### Similarité cosinus entre parfums

In [21]:
from sklearn.metrics.pairwise import cosine_similarity

# comparaison de chaque parfum avec les autres
similarites = cosine_similarity(vecteurs_parfums)

/Users/lisaroubertieperso/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/lisaroubertieperso/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/lisaroubertieperso/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b


### Première recommendation sur base d'un parfum

In [22]:
def recommander_parfums(indice, n=5):
    scores = similarites[indice]

    indices = scores.argsort()[::-1][1:n+1]

    resultats = df.iloc[indices][["name", "house"]].copy()
    resultats["similarite"] = scores[indices]

    return resultats

# test
print("Parfum choisi :", df.iloc[0]["name"])
recommander_parfums(0)

Parfum choisi : Libre


,name,house,similarite
752,Connoisseur Woman,Armaf,0.971148
337,Mon Guerlain,Guerlain,0.963083
1445,Valentino Voce Viva Intensa,Valentino,0.956364
1361,Olfactories Nue Au Soleil,Prada,0.952614
1226,La vie est belle L'Éclat,Lancôme,0.950763


In [23]:
print(df.iloc[0])

id                                                          ysl-libre
name                                                            Libre
house                                              Yves Saint Laurent
url                 https://www.scento.com/perfumes/libre-yves-sai...
family                                                    White Bloom
secondary_family                                                  NaN
character                              floral; creamy; fresh; citrusy
top_notes                                  mandarin; neroli; lavender
heart_notes                            orange blossom; jasmine sambac
base_notes                                              musk; vanilla
notes               mandarin; neroli; lavender; orange blossom; ja...
concentration                                           eau-de-parfum
intensity                                                         3.0
seasons                                        spring; summer; autumn
times_of_day        

In [24]:
print(df.iloc[752])

id                                            armaf-connoisseur-woman
name                                                Connoisseur Woman
house                                                           Armaf
url                 https://www.scento.com/perfume-database/perfum...
family                                                    White Bloom
secondary_family                                                  NaN
character                               floral; creamy; warm; citrusy
top_notes                                    bergamot; mandarin; pear
heart_notes                    orange blossom; jasmine sambac; neroli
base_notes                  musk; vanilla; benzoin; amber; tonka bean
notes               bergamot; mandarin; pear; orange blossom; jasm...
concentration                                           eau-de-parfum
intensity                                                         3.0
seasons                                        spring; summer; autumn
times_of_day        

In [25]:
print(df.iloc[337])

id                                              guerlain-mon-guerlain
name                                                     Mon Guerlain
house                                                        Guerlain
url                 https://www.scento.com/perfumes/mon-guerlain-g...
family                                                    White Bloom
secondary_family                                        Sunlit Citrus
character                              floral; fresh; citrusy; creamy
top_notes                                    bergamot; mandarin; pear
heart_notes                          neroli; jasmine sambac; lavender
base_notes                                  iris; vanilla; sandalwood
notes               bergamot; mandarin; pear; neroli; jasmine samb...
concentration                                           eau-de-parfum
intensity                                                         3.0
seasons                                        spring; summer; autumn
times_of_day        

Avec ce premier test, on vérifie que sur la base d'un parfum, le modèle est capable de retrouver des parfums similaires. En effet, en regardant les deux premières recommandations de similarité, on constate que le Libre, le Connoisseur Woman et le Mon Guerlain ont des notes similaires : famille White Bloom, les mêmes notes, le même moment de la journée où le porter... 

Le modèle semble bon sur les recommandations.

### Première recommandation sur base d'un profil structuré

On peut maintenant essayer de faire une recommandation de parfum avec une entrée utilisateur simulée comme elle le serait dans notre future interface.

In [26]:
# Entree utilisateur
notes_aimees = ["vanilla", "jasmine", "musk"]
notes_detestees = ["patchouli"]
moment = "evening"

# Vecteur de preference avec les notes appreciees
vecteurs_notes = [model.wv[note] for note in notes_aimees]
vecteur_utilisateur = np.mean(vecteurs_notes, axis=0)
print("Dimensions du profil utilisateur :", vecteur_utilisateur.shape)

# Comparaison entre le profil et les parfums
scores = cosine_similarity(vecteur_utilisateur.reshape(1, -1),vecteurs_parfums)[0]

# On ajoute les critères du questionnaire
parfum_soiree = df["time_evening"] == 1
parfum_patchouli = df[["top_notes", "heart_notes", "base_notes"]].apply(lambda colonne: colonne.str.contains("patchouli", case=False, na=False)).any(axis=1)
parfum_final = parfum_soiree & ~parfum_patchouli # le ~ permet d'inverser -> on garde les parfums qui ne contiennent pas de patchoulis

# Recommandations finales
resultats = df.loc[parfum_final, ["name", "house"]].copy()
resultats["similarite"] = scores[parfum_final]
resultats = resultats.sort_values(by="similarite", ascending=False)
resultats.head(10)

Dimensions du profil utilisateur : (100,)


/Users/lisaroubertieperso/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/lisaroubertieperso/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/lisaroubertieperso/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b


,name,house,similarite
679,Supremacy Gold,Afnan,0.838162
1135,Cruel Gardénia,Guerlain,0.819757
1036,Wind Flowers,Creed,0.817125
671,Rare Passion,Afnan,0.814901
888,Vanity Femme Essence Woman,Armaf,0.809175
1139,Samsara,Guerlain,0.804773
840,Orchestra Legato Lux,Armaf,0.803132
1281,Vanilla Extasy :,Montale,0.802868
913,My Way Intense,Giorgio Armani,0.802562
1302,all of me intense,Narciso Rodriguez,0.801681


In [27]:
# Verifications
verification = df.loc[resultats.head(10).index, ["name", "top_notes", "heart_notes", "base_notes", "time_evening"]].copy()
verification["similarite"] = resultats.head(10)["similarite"]
verification

,name,top_notes,heart_notes,base_notes,time_evening,similarite
679,Supremacy Gold,ylang-ylang; neroli,vetiver; musk; vanilla,jasmine; musk; sandalwood,1,0.838162
1135,Cruel Gardénia,rose; peach; neroli,gardenia; ylang-ylang; musk,tonka bean; vanilla; sandalwood,1,0.819757
1036,Wind Flowers,jasmine; orange blossom; peach,jasmine; tuberose; rose,musk; praline; sandalwood; iris,1,0.817125
671,Rare Passion,orange; musk; rose,amber; wood; vanilla,rose; musk; orange blossom,1,0.814901
888,Vanity Femme Essence Woman,freesia; orange blossom; peach,jasmine; magnolia; rose,amber; musk; vanilla,1,0.809175
1139,Samsara,honeysuckle,jasmine; ylang-ylang,sandalwood; iris; tonka bean; vanilla,1,0.804773
840,Orchestra Legato Lux,citrus; lychee; peach; vanilla,peony; orange blossom; jasmine; vanilla,musk; oakmoss; woody notes,1,0.803132
1281,Vanilla Extasy :,bergamot; peach; apricot,lavender; jasmine; ylang-ylang,sandalwood; vanilla; benzoin; amber; musk,1,0.802868
913,My Way Intense,bergamot; orange blossom,tuberose; jasmine,cedarwood; vanilla; musk,1,0.802562
1302,all of me intense,iris; jasmine sambac,rose; geranium; tuberose,musk; vanilla; sandalwood,1,0.801681


Le premier test est concluant : les parfums choisis par le modèle sont bien des parfums de soir, contenant de la vanille et/ou du musc et/ou du jasmin, et ne contiennent pas de patchouli.

### Deuxième recommandation sur base d'un texte libre

In [31]:
profil_texte_libre = """I am looking for a warm and sweet fragrance for the evening. I love vanilla and jasmine, but I dislike patchouli."""

# Même prétraitement que pour les descriptions
mots_utilisateur = (profil_texte_libre.lower().replace(',', "").replace('.', "").replace(':', "").split())

# Suppression des stop words
mots_utilisateur = [mot for mot in mots_utilisateur if mot not in mots_inutiles]
mots_utilisateur = [mot for mot in mots_utilisateur if mot not in ENGLISH_STOP_WORDS]

# Conservation des mots connus de Word2vec
mots_connus = [mot for mot in mots_utilisateur if mot in model.wv]
mots_inconnus = [mot for mot in mots_utilisateur if mot not in model.wv]

print('Mots après prétraitement : ', mots_utilisateur)
print('Mots connus : ', mots_connus)
print('Mots inconnus : ', mots_inconnus)

# Vecteur du texte utilisateur
vecteur_utilisateur_texte = np.mean([model.wv[mot] for mot in mots_connus], axis=0)

scores_texte = cosine_similarity(vecteur_utilisateur_texte.reshape(1, -1), vecteurs_parfums)[0]

resultats_texte = df[['name', 'house', 'top_notes', 'base_notes', 'heart_notes']].copy()
resultats_texte['similarite'] = scores_texte

resultats_texte = resultats_texte.sort_values(by='similarite', ascending=False)

resultats_texte.head(10)

Mots après prétraitement :  ['looking', 'warm', 'sweet', 'fragrance', 'evening', 'love', 'vanilla', 'jasmine', 'dislike', 'patchouli']
Mots connus :  ['warm', 'sweet', 'fragrance', 'evening', 'vanilla', 'jasmine', 'patchouli']
Mots inconnus :  ['looking', 'love', 'dislike']


/Users/lisaroubertieperso/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/lisaroubertieperso/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/lisaroubertieperso/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b


,name,house,top_notes,base_notes,heart_notes,similarite
1137,Mon Guerlain Intense,Guerlain,bergamot; mandarin; lavender,musk; patchouli; sandalwood,jasmine sambac; vanilla,0.877360
125,Red Aoud,Montale,bergamot; lemon; orange; lavender,vanilla; toffee; tonka bean; oud; musk; amber,rose; violet; patchouli,0.876446
886,Uniq Effects of Uniq,Armaf,pear; mandarin,coffee; caramel; cedarwood; patchouli; musk,vanilla; orange blossom,0.876144
646,9 PM Rebel,Afnan,mandarin; pineapple; apple,caramel; wood; ambergris; musk,cedarwood; oakmoss; vanilla,0.873728
1031,Royal Princess Oud,Creed,bergamot; rose,benzoin; sandalwood,jasmine; vanilla,0.873154
1444,Valentina,Valentino,bergamot; strawberry,cedarwood; vanilla,jasmine; patchouli,0.871452
997,Nomade Jasmin Naturel Intense,Chloé,jasmine,patchouli,dates,0.870398
1255,Feminity,Mancera,mandarin; pink pepper; coffee; leather,vanilla; amber; sandalwood; toffee; musk; oakmoss,violet; jasmine; lily of the valley,0.869098
692,Aurum Elixir,Ajmal,bergamot; blackcurrant; strawberry,jasmine; cashmere wood; caramel,patchouli; vanilla,0.869050
730,Blue Sky,Armaf,pineapple; apple; blackcurrant; pink pepper; b...,amber; vanilla; ambroxan; tobacco; cedarwood; ...,jasmine sambac; patchouli,0.867574


Le modèle Word2Vec nous montre sa limite ici. En effet, il récupère simplement les vecteurs des mots connus et en calcule une moyenne. Le problème ici est qu'il ne garde que les mots connus de son vocabulaire puis en calcule la moyenne pour représenter un profil.
Cependant, il ne prend pas en compte le sens des phrases ni les relations entre mots : il ne différencie pas les mots aimer et détester par exemple. Ainsi, le premier parfum similaire trouvé contient du patchouli alors que nous n'en voulions pas.

### Conclusion

Pour conclure sur ce modèle, on peut dire que Word2Vec peut être adapté à certaines utilisations nécessitant seulement une similarité entre mots. Dans notre projet, il est bon sur le conseil de parfum similaire : c'est logique car il s'agit de trouver les notes similaires (donc vecteurs similaires) entre parfums.

Malheureusement, notre projet consiste en une recommandation contextuelle. Il faut donc prendre en compte le sens des mots comme le fait d'aimer ou de ne pas aimer. Nous le comparerons tout de même aux autres modèles pour garder le meilleur en tant que modèle final.
